# Stage 1: Individual Dataset EDA & Baseline Modeling

In this stage, we analyze the three primary datasets independently to establish baseline patterns:
1. **Rainfall Anomaly (Climate Shock)**: Subnational rainfall patterns in Bangladesh.
2. **Food Price Volatility**: Trends and clustering of essential commodity prices.
3. **External Debt Indicators**: Macro-economic debt trends and drivers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
import os

# Settings
sns.set(style="whitegrid")
os.makedirs('production_artifacts/figures', exist_ok=True)

PCODE_MAP = {
    'BD10': 'Barisal', 'BD20': 'Chittagong', 'BD30': 'Dhaka', 
    'BD40': 'Khulna', 'BD50': 'Rajshahi', 'BD55': 'Rangpur', 
    'BD60': 'Sylhet', 'BD45': 'Mymensingh'
}

## 1.1 Dataset A: Rainfall Analysis
Focusing on subnational rainfall anomalies (RFQ) and clustering regions by climate risk.

In [ ]:
# Load Rainfall Data
rf_df = pd.read_csv('datasets/bgd-rainfall-subnat-full.csv')
rf_df['date'] = pd.to_datetime(rf_df['date'])
rf_df['year'] = rf_df['date'].dt.year

# Region-level Stats
region_stats = rf_df.groupby('PCODE')['rfq'].agg(['mean', 'std']).reset_index()
scaler = StandardScaler()
X_cluster = scaler.fit_transform(region_stats[['mean', 'std']])

# Clustering Districts by Climate Risk
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
region_stats['cluster'] = kmeans.fit_predict(X_cluster)
region_stats['region_name'] = region_stats['PCODE'].map(PCODE_MAP)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=region_stats, x='mean', y='std', hue='cluster', palette='viridis', s=100)
plt.title('Rainfall Clusters: Mean Anomaly vs Variability per Region')
plt.xlabel('Mean RFQ')
plt.ylabel('Standard Deviation of RFQ')
plt.show()

# Baseline RF Model for RFQ
features = ['year', 'month']
X_rf = rf_df[['year', 'month']]
y_rf = rf_df['rfq']
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_rf, y_rf)

importances = pd.Series(rf_model.feature_importances_, index=features)
plt.figure(figsize=(8, 4))
importances.plot(kind='barh')
plt.title('Feature Importance: Predicting Rainfall Anomaly')
plt.show()

## 1.2 Dataset B: Food Price Volatility
Analyzing price trends for essential commodities like Rice, Lentils, and Oil.

In [ ]:
# Load Food Price Data
food_df = pd.read_csv('datasets/wfp_food_prices_bgd.csv')
food_df['date'] = pd.to_datetime(food_df['date'])

# Market Volatility Clustering
market_stats = food_df.groupby('market')['price'].agg(['mean', 'std']).dropna()
X_market = scaler.fit_transform(market_stats)
market_stats['cluster'] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_market)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=market_stats, x='mean', y='std', hue='cluster', palette='magma')
plt.title('Market Clusters: Price Level vs Volatility')
plt.show()

# Time-Series Prediction for Rice (National Average)
rice_df = food_df[food_df['commodity'] == 'Rice (paddy)'].groupby('date')['price'].mean().reset_index()
rice_df['year'] = rice_df['date'].dt.year
rice_df['month'] = rice_df['date'].dt.month

X_rice = rice_df[['year', 'month']]
y_rice = rice_df['price']

gb_model = GradientBoostingRegressor(n_estimators=100)
gb_model.fit(X_rice, y_rice)
rice_df['pred'] = gb_model.predict(X_rice)

plt.figure(figsize=(12, 5))
plt.plot(rice_df['date'], rice_df['price'], label='Actual')
plt.plot(rice_df['date'], rice_df['pred'], label='Model', linestyle='--')
plt.title('Rice Price Forecasting: Gradient Boosting Baseline')
plt.legend()
plt.show()

## 1.3 Dataset C: External Debt Indicators
Analyzing macro-level debt drivers including imports, exports, and remittances.

In [ ]:
# Load External Debt Data
debt_df = pd.read_csv('datasets/external-debt_bgd.csv')

relevant_indicators = {
    'External debt stocks, total (DOD, current US$)': 'total_external_debt',
    'Imports of goods, services and primary income (BoP, current US$)': 'imports',
    'Exports of goods, services and primary income (BoP, current US$)': 'exports',
    'Current account balance (BoP, current US$)': 'current_account_balance',
    'Personal remittances, received (current US$)': 'personal_remittances',
    'Foreign direct investment, net inflows (BoP, current US$)': 'fdi_inflows'
}

df_filtered = debt_df[debt_df['Indicator Name'].isin(relevant_indicators.keys())].copy()
df_filtered['Indicator Name'] = df_filtered['Indicator Name'].map(relevant_indicators)

wide_df = df_filtered.pivot_table(index='Year', columns='Indicator Name', values='Value').reset_index()
wide_df.rename(columns={'Year': 'year'}, inplace=True)
wide_df.dropna(inplace=True)

plt.figure(figsize=(10, 6))
plt.plot(wide_df['year'], wide_df['total_external_debt'] / 1e9, marker='o', color='red')
plt.title('Total External Debt Trend (Billion USD)')
plt.ylabel('Debt Stock (Billion $)')
plt.show()

# Regression Analysis of Debt Drivers
features = ['imports', 'exports', 'current_account_balance', 'personal_remittances']
X_debt = scaler.fit_transform(wide_df[features])
y_debt = wide_df['total_external_debt']

ridge = Ridge(alpha=1.0)
ridge.fit(X_debt, y_debt)

coef_df = pd.DataFrame({'Feature': features, 'Coef': ridge.coef_})
plt.figure(figsize=(8, 4))
sns.barplot(data=coef_df, x='Coef', y='Feature')
plt.title('Ridge Regression Coefficients: Drivers of External Debt')
plt.show()